# Phase 5: Targeted Analysis 5: Edge Importance Profiling

## Overview

NB01-NB04 established WHAT non-universal edges do (they boost the universal core generically,
not band-specifically). This notebook asks WHERE they are: which layers, heads, and edge types
contain universal vs non-universal edges?

## Key Questions

1. **Layer profile**: Do non-universal edges cluster in early, middle, or late layers?
2. **Head universality**: Which attention heads are most/least universal?
3. **Reliable band-specific**: How many edges are reliably band-specific (consistent across draws)?
4. **Per-example failures**: What examples does the universal core fail on?

## Method (CPU only: no GPU evals needed)

1. Load all 60 prune_scores, classify each edge position by sharing level (0-5 bands)
2. Decompose by layer, head, edge type
3. Cross-reference with Phase 2 data
4. Analyze per-example failures

## Sections

1. Edge Sharing Distribution
2. Layer-Level Profile
3. Head-Level Analysis
4. Reliable Band-Specific Edges
5. Per-Example Failure Patterns
6. Summary

## Data Sources

- 60 circuits: 'circuit_discovery/circuits/{model}/{band}/{draw}/prune_scores.pkl'
- Phase 2 CSVs: '02_Phase_Structural/outputs/analysis/'
- Per-example eval: 'LSC_circuits/per_example_eval/per_example_summary.csv'

In [1]:
import os
import sys
import pickle
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch as t
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

# Paths
ANALYSIS_ROOT = Path("LSC_circuit_analysis")
ISC_ROOT = Path(os.environ.get("PROJECT_ROOT", ".")).resolve()
LSC_DIR = ISC_ROOT / "LSC_circuits"
CIRCUITS_DIR = LSC_DIR / "circuit_discovery" / "circuits"

PHASE2_DIR = ANALYSIS_ROOT / "02_Phase_Structural" / "outputs" / "analysis"
PHASE5_DIR = ANALYSIS_ROOT / "05_Phase_Targeted"
ANALYSIS_DIR = PHASE5_DIR / "outputs" / "analysis"
VIZ_DIR = PHASE5_DIR / "outputs" / "viz"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
VIZ_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(LSC_DIR))
from lsc_acdc_circuit import model_safe_name

# Constants
MODELS = ["pythia-70m", "pythia-160m", "pythia-410m", "pythia-1b", "pythia-1.4b"]
BANDS = ["low", "medium", "high", "very_high", "control"]
DRAWS = ["draw_1", "draw_2", "draw_3"]

# Model architecture info
MODEL_LAYERS = {
    "pythia-70m": 6,
    "pythia-160m": 12,
    "pythia-410m": 24,
    "pythia-1b": 16,
    "pythia-1.4b": 24,
}
MODEL_HEADS = {
    "pythia-70m": 8,
    "pythia-160m": 12,
    "pythia-410m": 16,
    "pythia-1b": 8,
    "pythia-1.4b": 16,
}

# Plot defaults
sns.set_theme(style="whitegrid", font_scale=1.1)
BAND_COLORS = {
    "low": "#2196F3",
    "medium": "#4CAF50",
    "high": "#FF9800",
    "very_high": "#F44336",
    "control": "#9E9E9E",
}


def save_figure(fig, filename):
    path = VIZ_DIR / filename
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {path}")


def parse_module_name(module_name):
    """Extract layer and edge type from auto_circuit module name.
    Note: head info is NOT in the module name: it's in the tensor's first dim
    for attn modules (shape [n_heads, n_sources]).

    Examples:
      'blocks.0.hook_attn_in' -> layer=0, type='attn'
      'blocks.0.hook_mlp_in' -> layer=0, type='mlp'
      'blocks.5.hook_resid_post' -> layer=5, type='resid'
    """
    layer_match = re.search(r"blocks\.(\d+)", module_name)
    layer = int(layer_match.group(1)) if layer_match else None

    if "attn" in module_name:
        edge_type = "attn"
    elif "mlp" in module_name:
        edge_type = "mlp"
    elif "resid" in module_name:
        edge_type = "resid"
    elif "embed" in module_name.lower():
        edge_type = "embed"
    else:
        edge_type = "other"

    return layer, edge_type


print("Setup complete.")

Setup complete.


## 1. Edge Sharing Distribution

For each model x draw: load all 5 bands' binary masks, sum per edge position.
How many edges are in 5/5 bands (universal), 4/5, ..., 1/5?

In [2]:
# Compute sharing counts per model x draw
sharing_data = []  # list of {model, draw, module, position, sharing_count, layer, head, type}
sharing_histograms = []  # summary rows

for model_name in MODELS:
    m_safe = model_safe_name(model_name)
    print(f"\n{'=' * 60}")
    print(f"Model: {model_name}")
    print(f"{'=' * 60}")

    for draw in DRAWS:
        # Load all 5 bands
        all_band_scores = {}
        missing = False
        for band in BANDS:
            scores_path = CIRCUITS_DIR / m_safe / band / draw / "prune_scores.pkl"
            if not scores_path.exists():
                missing = True
                break
            with open(scores_path, "rb") as f:
                all_band_scores[band] = pickle.load(f)
        if missing:
            continue

        # Compute sharing count per edge position
        first_band = BANDS[0]
        level_counts = {level: 0 for level in range(6)}

        for module_name in all_band_scores[first_band]:
            count = t.zeros_like(
                all_band_scores[first_band][module_name], dtype=t.int32
            )
            for band in BANDS:
                count += t.isinf(all_band_scores[band][module_name]).int()

            layer, edge_type = parse_module_name(module_name)

            # Count at each level
            flat = count.flatten()
            for level in range(6):
                n = (flat == level).sum().item()
                level_counts[level] += n

            # For attn modules (shape [n_heads, n_sources]): extract per-head info
            # For mlp/resid (shape [n_sources]): head=None
            if edge_type == "attn" and count.dim() == 2:
                n_heads = count.shape[0]
                for head_idx in range(n_heads):
                    head_row = count[head_idx]
                    active_mask = head_row > 0
                    if active_mask.sum() == 0:
                        continue
                    active_positions = active_mask.nonzero(as_tuple=True)[0]
                    active_counts = head_row[active_mask]
                    for pos, cnt in zip(
                        active_positions.tolist(), active_counts.tolist()
                    ):
                        sharing_data.append(
                            {
                                "model": model_name,
                                "draw": draw,
                                "module": module_name,
                                "position": pos,
                                "sharing_count": cnt,
                                "layer": layer,
                                "head": head_idx,
                                "edge_type": edge_type,
                            }
                        )
            else:
                # mlp, resid, or 1D attn: no head dimension
                active_mask = flat > 0
                active_positions = active_mask.nonzero(as_tuple=True)[0]
                active_counts = flat[active_mask]
                for pos, cnt in zip(active_positions.tolist(), active_counts.tolist()):
                    sharing_data.append(
                        {
                            "model": model_name,
                            "draw": draw,
                            "module": module_name,
                            "position": pos,
                            "sharing_count": cnt,
                            "layer": layer,
                            "head": None,
                            "edge_type": edge_type,
                        }
                    )

        sharing_histograms.append(
            {
                "model": model_name,
                "draw": draw,
                **{f"level_{level}": level_counts[level] for level in range(6)},
            }
        )

        in_any = sum(level_counts[l] for l in range(1, 6))
        print(
            f"  {draw}: "
            + ", ".join(f"={l}:{level_counts[l]}" for l in range(6))
            + f" (total active: {in_any})"
        )

        del all_band_scores

df_sharing = pd.DataFrame(sharing_data)
df_hist = pd.DataFrame(sharing_histograms)
print(f"\nSharing data: {len(df_sharing)} edge positions across all models/draws")
print(f"Histograms: {len(df_hist)} rows")
print(f"Attn edges with head info: {df_sharing[df_sharing['head'].notna()].shape[0]}")


Model: pythia-70m
  draw_1: =0:789, =1:80, =2:50, =3:49, =4:59, =5:297 (total active: 535)
  draw_2: =0:798, =1:78, =2:51, =3:45, =4:55, =5:297 (total active: 526)
  draw_3: =0:787, =1:73, =2:61, =3:47, =4:45, =5:311 (total active: 537)

Model: pythia-160m


  draw_1: =0:9131, =1:638, =2:409, =3:287, =4:308, =5:694 (total active: 2336)
  draw_2: =0:9072, =1:671, =2:416, =3:322, =4:263, =5:723 (total active: 2395)


  draw_3: =0:9206, =1:583, =2:398, =3:279, =4:278, =5:723 (total active: 2261)

Model: pythia-410m
  draw_1: =0:73391, =1:2800, =2:1443, =3:955, =4:709, =5:1283 (total active: 7190)


  draw_2: =0:73383, =1:2809, =2:1438, =3:957, =4:693, =5:1301 (total active: 7198)


  draw_3: =0:73357, =1:2767, =2:1427, =3:982, =4:724, =5:1324 (total active: 7224)

Model: pythia-1b
  draw_1: =0:8259, =1:584, =2:355, =3:263, =4:200, =5:348 (total active: 1750)
  draw_2: =0:8268, =1:617, =2:308, =3:258, =4:202, =5:356 (total active: 1741)
  draw_3: =0:8309, =1:576, =2:342, =3:247, =4:171, =5:364 (total active: 1700)

Model: pythia-1.4b


  draw_1: =0:75496, =1:2279, =2:1042, =3:651, =4:488, =5:625 (total active: 5085)
  draw_2: =0:75403, =1:2358, =2:1064, =3:658, =4:482, =5:616 (total active: 5178)
  draw_3: =0:75502, =1:2310, =2:1052, =3:609, =4:489, =5:619 (total active: 5079)



Sharing data: 50735 edge positions across all models/draws
Histograms: 15 rows
Attn edges with head info: 33451


### VIZ 01: Sharing Distribution

Histogram of edge sharing levels per model. Shows how many edges are universal (5/5) vs partially shared.

In [3]:
fig, axes = plt.subplots(1, len(MODELS), figsize=(5 * len(MODELS), 5))

level_colors = ["#BBDEFB", "#90CAF9", "#42A5F5", "#1E88E5", "#0D47A1"]

for ax, model_name in zip(axes, MODELS):
    sub = df_hist[df_hist["model"] == model_name]

    # Mean across draws (only levels 1-5, excluding 0)
    means = [sub[f"level_{l}"].mean() for l in range(1, 6)]
    stds = [sub[f"level_{l}"].std() for l in range(1, 6)]

    x = range(5)
    bars = ax.bar(
        x,
        means,
        yerr=stds,
        color=level_colors,
        edgecolor="#333",
        linewidth=0.5,
        capsize=4,
    )

    # Annotate with count and percentage
    total = sum(means)
    for i, (m, s) in enumerate(zip(means, stds)):
        pct = m / total * 100 if total > 0 else 0
        ax.text(
            i,
            m + s + max(means) * 0.02,
            f"{m:.0f}\n({pct:.1f}%)",
            ha="center",
            va="bottom",
            fontsize=8,
        )

    ax.set_xlabel("Sharing level (in N of 5 bands)")
    ax.set_xticks(range(5))
    ax.set_xticklabels(["1", "2", "3", "4", "5"])
    ax.set_title(model_name, fontsize=12, fontweight="bold")

axes[0].set_ylabel("Number of edge positions")
fig.suptitle(
    "Edge Sharing Distribution: How many edges appear in N of 5 bands?\n"
    "(excluding level 0 = never selected, mean +/- std across 3 draws)",
    fontsize=13,
    fontweight="bold",
)
fig.tight_layout()
save_figure(fig, "T5_01_sharing_distribution.png")

Saved: LSC_circuit_analysis/05_Phase_Targeted/outputs/viz/T5_01_sharing_distribution.png


## 2. Layer-Level Profile

For each model x layer: what fraction of edges at each sharing level?
Shows whether non-universal edges cluster in specific layers.

In [4]:
# Compute layer x sharing level profiles
layer_profile_rows = []

for model_name in MODELS:
    sub = df_sharing[df_sharing["model"] == model_name]
    # Filter to edges with layer info
    sub_with_layer = sub[sub["layer"].notna()].copy()
    sub_with_layer["layer"] = sub_with_layer["layer"].astype(int)

    n_layers = MODEL_LAYERS[model_name]

    for layer in range(n_layers):
        layer_sub = sub_with_layer[sub_with_layer["layer"] == layer]
        total_in_layer = len(layer_sub)

        for level in range(1, 6):
            n_at_level = len(layer_sub[layer_sub["sharing_count"] == level])
            frac = n_at_level / total_in_layer if total_in_layer > 0 else 0

            layer_profile_rows.append(
                {
                    "model": model_name,
                    "layer": layer,
                    "sharing_level": level,
                    "n_edges": n_at_level,
                    "fraction": frac,
                    "total_in_layer": total_in_layer,
                }
            )

df_layer_profile = pd.DataFrame(layer_profile_rows)
df_layer_profile.to_csv(ANALYSIS_DIR / "layer_head_sharing.csv", index=False)
print(f"Saved: layer_head_sharing.csv ({len(df_layer_profile)} rows)")

Saved: layer_head_sharing.csv (410 rows)


### VIZ 02: Layer Profile Heatmaps

Per model: heatmap with x=sharing level (1-5), y=layer. Cell value = fraction of edges at that level.

In [5]:
for model_name in MODELS:
    sub = df_layer_profile[df_layer_profile["model"] == model_name]
    pivot = sub.pivot(index="layer", columns="sharing_level", values="fraction")
    pivot = pivot.sort_index(ascending=True)

    n_layers = MODEL_LAYERS[model_name]
    fig_height = max(4, n_layers * 0.45)
    fig, ax = plt.subplots(figsize=(6, fig_height))

    sns.heatmap(
        pivot,
        annot=True,
        fmt=".2f",
        cmap="YlOrRd",
        vmin=0,
        vmax=1,
        ax=ax,
        linewidths=0,
        linecolor="none",
        xticklabels=[f"{l}/5" for l in range(1, 6)],
        cbar_kws={"shrink": 0.8},
    )

    ax.set_xlabel("Sharing level (in N/5 bands)")
    ax.set_ylabel("Layer")
    ax.set_title(
        f"{model_name}: Layer x Sharing Level Profile\n"
        "(fraction of active edges at each sharing level, mean across 3 draws)",
        fontsize=11,
        fontweight="bold",
    )

    fig.tight_layout()
    save_figure(fig, f"T5_02_layer_profile_{model_name}.png")

Saved: LSC_circuit_analysis/05_Phase_Targeted/outputs/viz/T5_02_layer_profile_pythia-70m.png


Saved: LSC_circuit_analysis/05_Phase_Targeted/outputs/viz/T5_02_layer_profile_pythia-160m.png


Saved: LSC_circuit_analysis/05_Phase_Targeted/outputs/viz/T5_02_layer_profile_pythia-410m.png


Saved: LSC_circuit_analysis/05_Phase_Targeted/outputs/viz/T5_02_layer_profile_pythia-1b.png


Saved: LSC_circuit_analysis/05_Phase_Targeted/outputs/viz/T5_02_layer_profile_pythia-1.4b.png


## 3. Head-Level Analysis

Load Phase 2 head universality data and visualize which heads are most/least universal.

In [6]:
# Try loading Phase 2 head universality data
head_univ_path = PHASE2_DIR / "deep_head_universality.csv"
head_entropy_path = PHASE2_DIR / "deep_head_entropy.csv"

if head_univ_path.exists():
    df_head_univ = pd.read_csv(head_univ_path)
    print(f"Loaded head universality: {len(df_head_univ)} rows")
    print(df_head_univ.head())
else:
    print(f"WARNING: {head_univ_path} not found")
    df_head_univ = None

if head_entropy_path.exists():
    df_head_entropy = pd.read_csv(head_entropy_path)
    print(f"\nLoaded head entropy: {len(df_head_entropy)} rows")
else:
    print(f"WARNING: {head_entropy_path} not found")
    df_head_entropy = None

# Also compute head-level sharing from our data
head_sharing_rows = []
for model_name in MODELS:
    sub = df_sharing[
        (df_sharing["model"] == model_name)
        & (df_sharing["edge_type"] == "attn")
        & (df_sharing["head"].notna())
    ].copy()
    sub["layer"] = sub["layer"].astype(int)
    sub["head"] = sub["head"].astype(int)

    for (layer, head), grp in sub.groupby(["layer", "head"]):
        total = len(grp)
        mean_sharing = grp["sharing_count"].mean()
        frac_universal = (grp["sharing_count"] == 5).sum() / total if total > 0 else 0

        head_sharing_rows.append(
            {
                "model": model_name,
                "layer": layer,
                "head": head,
                "n_edges": total,
                "mean_sharing": mean_sharing,
                "frac_universal": frac_universal,
            }
        )

df_head_sharing = pd.DataFrame(head_sharing_rows)
print(f"\nComputed head sharing: {len(df_head_sharing)} heads")

Loaded head universality: 3264 rows
        model    draw  head  layer  head_idx  n_bands  \
0  pythia-70m  draw_1  A0.0      0         0        5   
1  pythia-70m  draw_1  A0.1      0         1        5   
2  pythia-70m  draw_1  A0.2      0         2        5   
3  pythia-70m  draw_1  A0.3      0         3        5   
4  pythia-70m  draw_1  A0.4      0         4        5   

                       bands_present  universality_score  
0  low,medium,high,very_high,control                 1.0  
1  low,medium,high,very_high,control                 1.0  
2  low,medium,high,very_high,control                 1.0  
3  low,medium,high,very_high,control                 1.0  
4  low,medium,high,very_high,control                 1.0  

Loaded head entropy: 3264 rows



Computed head sharing: 874 heads


### VIZ 03: Head Universality Map

Per model: grid of heads (layer x head_idx), colored by mean sharing level.
Highlights which heads are most/least universal.

In [7]:
for model_name in MODELS:
    sub = df_head_sharing[df_head_sharing["model"] == model_name]
    if len(sub) == 0:
        print(f"{model_name}: No data, skipping.")
        continue

    n_layers = MODEL_LAYERS[model_name]
    n_heads = MODEL_HEADS[model_name]

    # Build matrix
    matrix = np.full((n_layers, n_heads), np.nan)
    for _, row in sub.iterrows():
        l, h = int(row["layer"]), int(row["head"])
        if l < n_layers and h < n_heads:
            matrix[l, h] = row["mean_sharing"]

    fig_width = max(5, n_heads * 0.55)
    fig_height = max(4, n_layers * 0.45)
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    fig.patch.set_facecolor("white")
    ax.set_facecolor("white")

    sns.heatmap(
        matrix,
        annot=False,
        cmap="RdYlGn",
        vmin=1,
        vmax=5,
        ax=ax,
        linewidths=0,
        linecolor="none",
        square=True,
        cbar_kws={"shrink": 0.8, "label": "Mean sharing"},
    )

    ax.grid(False)
    ax.set_xlabel("Head index")
    ax.set_ylabel("Layer")
    ax.set_title(
        f"{model_name}: Head Universality Map\n"
        "(5.0 = all edges universal, 1.0 = all band-specific, NaN = no edges)",
        fontsize=11,
        fontweight="bold",
    )

    fig.tight_layout()
    save_figure(fig, f"T5_03_head_universality_{model_name}.png")

Saved: LSC_circuit_analysis/05_Phase_Targeted/outputs/viz/T5_03_head_universality_pythia-70m.png


Saved: LSC_circuit_analysis/05_Phase_Targeted/outputs/viz/T5_03_head_universality_pythia-160m.png


Saved: LSC_circuit_analysis/05_Phase_Targeted/outputs/viz/T5_03_head_universality_pythia-410m.png


Saved: LSC_circuit_analysis/05_Phase_Targeted/outputs/viz/T5_03_head_universality_pythia-1b.png


Saved: LSC_circuit_analysis/05_Phase_Targeted/outputs/viz/T5_03_head_universality_pythia-1.4b.png


## 4. Reliable Band-Specific Edges

Load Phase 2 reliable band-specific edge data. How many edges are reliably band-specific
(present in same band across all 3 draws, but NOT in other bands)?

In [8]:
# Load Phase 2 reliable band-specific data
rbs_path = PHASE2_DIR / "deep_reliable_band_specific.csv"
if rbs_path.exists():
    df_rbs = pd.read_csv(rbs_path)
    print(f"Loaded reliable band-specific: {len(df_rbs)} rows")
    print(df_rbs.head(20))
else:
    print(f"WARNING: {rbs_path} not found")
    df_rbs = None

# Also compute from our sharing data:
# An edge is "reliably band-specific" if it appears in exactly 1 band
# AND that's consistent across all 3 draws
# (sharing_count == 1 across all draws for the same modulexposition)

# Simpler approach: count edges at sharing_count == 1 per draw, see consistency
band_specific_counts = []
for model_name in MODELS:
    for draw in DRAWS:
        sub = df_sharing[
            (df_sharing["model"] == model_name) & (df_sharing["draw"] == draw)
        ]
        n_total = len(sub)
        n_level1 = len(sub[sub["sharing_count"] == 1])
        n_level5 = len(sub[sub["sharing_count"] == 5])
        band_specific_counts.append(
            {
                "model": model_name,
                "draw": draw,
                "n_total_active": n_total,
                "n_in_1_band": n_level1,
                "n_in_5_bands": n_level5,
                "frac_in_1_band": n_level1 / n_total if n_total > 0 else 0,
                "frac_universal": n_level5 / n_total if n_total > 0 else 0,
            }
        )

df_bsc = pd.DataFrame(band_specific_counts)
print("\n--- Band-specific (sharing=1) vs Universal (sharing=5) ---")
for model_name in MODELS:
    sub = df_bsc[df_bsc["model"] == model_name]
    print(
        f"  {model_name}: "
        f"band-specific={sub['frac_in_1_band'].mean():.1%}, "
        f"universal={sub['frac_universal'].mean():.1%}"
    )

Loaded reliable band-specific: 438 rows
                              raw        model       band  n_draws  \
0     blocks.4.hook_attn_in[5,18]   pythia-70m        low        3   
1     blocks.4.hook_attn_in[6,32]   pythia-70m        low        3   
2     blocks.4.hook_attn_in[4,32]   pythia-70m     medium        3   
3     blocks.4.hook_attn_in[7,11]   pythia-70m       high        3   
4     blocks.5.hook_attn_in[3,33]   pythia-70m       high        3   
5        blocks.3.hook_mlp_in[17]   pythia-70m  very_high        3   
6     blocks.3.hook_attn_in[3,15]   pythia-70m  very_high        3   
7     blocks.3.hook_attn_in[4,20]   pythia-70m  very_high        3   
8      blocks.3.hook_attn_in[0,2]   pythia-70m    control        3   
9     blocks.4.hook_attn_in[0,18]   pythia-70m    control        3   
10     blocks.4.hook_attn_in[7,8]   pythia-70m    control        3   
11  blocks.10.hook_attn_in[10,99]  pythia-160m        low        3   
12      blocks.10.hook_mlp_in[46]  pythia-160m    

### VIZ 04: Reliable vs Spurious Band-Specific Edges

For each model: how many band-specific edges (sharing=1) are consistent across draws?
If most band-specific edges are inconsistent, they're ACDC noise.

In [9]:
fig, axes = plt.subplots(1, len(MODELS), figsize=(5 * len(MODELS), 5.5))

for ax, model_name in zip(axes, MODELS):
    sub = df_hist[df_hist["model"] == model_name]

    # Stacked bar: per draw, show composition
    draws = sub["draw"].values
    x = np.arange(len(draws))

    # Stack: universal (5), partially shared (2-4), band-specific (1)
    vals_5 = sub["level_5"].values
    vals_24 = sub[["level_2", "level_3", "level_4"]].sum(axis=1).values
    vals_1 = sub["level_1"].values

    ax.bar(x, vals_5, color="#1B5E20", alpha=0.85, label="Universal (5/5)")
    ax.bar(
        x,
        vals_24,
        bottom=vals_5,
        color="#FF9800",
        alpha=0.85,
        label="Partially shared (2-4/5)",
    )
    ax.bar(
        x,
        vals_1,
        bottom=vals_5 + vals_24,
        color="#D32F2F",
        alpha=0.85,
        label="Band-specific (1/5)",
    )

    # Annotate percentages
    for i in range(len(draws)):
        total = vals_5[i] + vals_24[i] + vals_1[i]
        if total > 0:
            ax.text(
                i,
                total + total * 0.02,
                f"{vals_1[i] / total:.0%}\nBS",
                ha="center",
                fontsize=7,
                color="darkred",
            )

    ax.set_xticks(x)
    ax.set_xticklabels(draws, rotation=45, ha="right")
    ax.set_title(model_name, fontsize=12, fontweight="bold")
    ax.legend(fontsize=7, loc="upper right")

axes[0].set_ylabel("Number of active edge positions")
fig.suptitle(
    "Edge Composition by Draw: Universal vs Partially Shared vs Band-Specific\n"
    "(red annotation = fraction of band-specific edges)",
    fontsize=13,
    fontweight="bold",
)
fig.tight_layout()
save_figure(fig, "T5_04_reliable_band_specific.png")

Saved: LSC_circuit_analysis/05_Phase_Targeted/outputs/viz/T5_04_reliable_band_specific.png


## 5. Per-Example Failure Patterns

Which examples does the full circuit get right but the universal core misses?
Are failures systematic (low-frequency tokens) or random?

In [10]:
# Load per-example evaluation data
per_example_path = Path("LSC_circuits/per_example_eval/per_example_summary.csv")

if per_example_path.exists():
    df_per_example = pd.read_csv(per_example_path)
    print(f"Loaded per-example data: {len(df_per_example)} rows")
    print(f"Columns: {list(df_per_example.columns)}")
    print(df_per_example.head())
else:
    print(f"WARNING: {per_example_path} not found")
    df_per_example = None

# Load NB01 universal core eval results for per-example analysis
univ_eval_path = ANALYSIS_DIR / "universal_core_eval_results.csv"
if univ_eval_path.exists():
    df_univ_eval = pd.read_csv(univ_eval_path)
    print(f"\nLoaded universal core eval: {len(df_univ_eval)} rows")
else:
    df_univ_eval = None

Loaded per-example data: 75 rows
Columns: ['model', 'train_band', 'draw', 'test_band', 'n_examples', 'accuracy', 'mean_correct_prob', 'std_correct_prob', 'median_correct_prob']
         model train_band    draw  test_band  n_examples  accuracy  \
0  pythia-1.4b        low  draw_1        low         225  0.817778   
1  pythia-1.4b        low  draw_1     medium         225  0.915556   
2  pythia-1.4b        low  draw_1       high         225  0.937778   
3  pythia-1.4b        low  draw_1  very_high         225  0.888889   
4  pythia-1.4b        low  draw_1    control         225  0.902222   

   mean_correct_prob  std_correct_prob  median_correct_prob  
0           0.401890          0.311157             0.331440  
1           0.455188          0.312530             0.390115  
2           0.454915          0.291365             0.444839  
3           0.392866          0.298035             0.334690  
4           0.419494          0.310342             0.377402  

Loaded universal core eval: 1

### VIZ 05: Per-Example Failure Patterns

Analyze failure patterns of the universal core. If per-example data exists,
compare full circuit vs universal core accuracy by band and frequency.

In [11]:
fig, axes = plt.subplots(1, len(MODELS), figsize=(5 * len(MODELS), 5.5))

# Use NB01 universal core comparison data
df_nb01 = pd.read_csv(ANALYSIS_DIR / "universal_core_comparison.csv")

for ax, model_name in zip(axes, MODELS):
    sub = df_nb01[df_nb01["model"] == model_name]

    x = np.arange(len(BANDS))
    width = 0.3

    full_accs = [
        sub[sub["test_band"] == b]["full_circuit_acc"].values[0] for b in BANDS
    ]
    univ_accs = [sub[sub["test_band"] == b]["universal_acc"].values[0] for b in BANDS]
    gaps = [f - u for f, u in zip(full_accs, univ_accs)]

    ax.bar(
        x - width / 2,
        full_accs,
        width,
        color="#1976D2",
        alpha=0.85,
        label="Full circuit",
    )
    ax.bar(
        x + width / 2,
        univ_accs,
        width,
        color="#388E3C",
        alpha=0.85,
        label="Universal core",
    )

    # Annotate gap
    for i, gap in enumerate(gaps):
        ax.annotate(
            f"gap={gap:.3f}",
            (i, min(full_accs[i], univ_accs[i]) - 0.02),
            ha="center",
            fontsize=7,
            color="red",
            fontweight="bold",
        )

    ax.set_xticks(x)
    ax.set_xticklabels(BANDS, rotation=45, ha="right")
    ax.set_title(model_name, fontsize=12, fontweight="bold")
    ax.legend(fontsize=8, loc="lower right")

axes[0].set_ylabel("Accuracy")
fig.suptitle(
    "Accuracy Gap: Full Circuit vs Universal Core per Band\n"
    "(gap = accuracy lost by removing non-universal edges)",
    fontsize=13,
    fontweight="bold",
)
fig.tight_layout()
save_figure(fig, "T5_05_failure_patterns.png")

Saved: LSC_circuit_analysis/05_Phase_Targeted/outputs/viz/T5_05_failure_patterns.png


## 6. Summary

In [12]:
print("=" * 80)
print("PHASE 5: TARGETED ANALYSIS 5: EDGE IMPORTANCE PROFILING")
print("=" * 80)

print("\n--- Edge Sharing Distribution (mean across draws) ---")
print(
    f"{'Model':<14} {'1/5':>8} {'2/5':>8} {'3/5':>8} {'4/5':>8} {'5/5':>8} {'Total':>8}"
)
print("-" * 62)
for model_name in MODELS:
    sub = df_hist[df_hist["model"] == model_name]
    vals = [sub[f"level_{l}"].mean() for l in range(1, 6)]
    total = sum(vals)
    print(
        f"{model_name:<14} " + " ".join(f"{v:>8.0f}" for v in vals) + f" {total:>8.0f}"
    )

print("\n--- As percentages ---")
print(f"{'Model':<14} {'1/5':>8} {'2/5':>8} {'3/5':>8} {'4/5':>8} {'5/5':>8}")
print("-" * 54)
for model_name in MODELS:
    sub = df_hist[df_hist["model"] == model_name]
    vals = [sub[f"level_{l}"].mean() for l in range(1, 6)]
    total = sum(vals)
    pcts = [v / total * 100 for v in vals]
    print(f"{model_name:<14} " + " ".join(f"{p:>7.1f}%" for p in pcts))

print("\n--- Head Universality (most/least universal heads) ---")
for model_name in MODELS:
    sub = df_head_sharing[df_head_sharing["model"] == model_name].sort_values(
        "mean_sharing"
    )
    if len(sub) == 0:
        print(f"\n  {model_name}: no head data available")
        continue
    least = sub.head(3)
    most = sub.tail(3)
    print(f"\n  {model_name}:")
    print(
        f"    Most universal: "
        + ", ".join(
            f"L{int(r['layer'])}H{int(r['head'])}({r['mean_sharing']:.1f})"
            for _, r in most.iterrows()
        )
    )
    print(
        f"    Least universal: "
        + ", ".join(
            f"L{int(r['layer'])}H{int(r['head'])}({r['mean_sharing']:.1f})"
            for _, r in least.iterrows()
        )
    )

print("\n--- Output Files ---")
for f in sorted(
    list(ANALYSIS_DIR.glob("*sharing*.csv")) + list(ANALYSIS_DIR.glob("layer_*.csv"))
):
    print(f"  {f.name} ({f.stat().st_size / 1024:.1f} KB)")
for f in sorted(VIZ_DIR.glob("T5_*.png")):
    print(f"  {f.name} ({f.stat().st_size / 1024:.1f} KB)")

print("\nDone.")

PHASE 5: TARGETED ANALYSIS 5: EDGE IMPORTANCE PROFILING

--- Edge Sharing Distribution (mean across draws) ---
Model               1/5      2/5      3/5      4/5      5/5    Total
--------------------------------------------------------------
pythia-70m           77       54       47       53      302      533
pythia-160m         631      408      296      283      713     2331
pythia-410m        2792     1436      965      709     1303     7204
pythia-1b           592      335      256      191      356     1730
pythia-1.4b        2316     1053      639      486      620     5114

--- As percentages ---
Model               1/5      2/5      3/5      4/5      5/5
------------------------------------------------------
pythia-70m        14.5%    10.1%     8.8%     9.9%    56.6%
pythia-160m       27.1%    17.5%    12.7%    12.1%    30.6%
pythia-410m       38.8%    19.9%    13.4%     9.8%    18.1%
pythia-1b         34.2%    19.4%    14.8%    11.0%    20.6%
pythia-1.4b       45.3%    20.6% 